In [10]:
# Importation des bibliothèques
from pathlib import Path
from datetime import datetime
import csv
import pandas as pd

# Dossier du projet : fonctionne si le notebook est à la racine du projet
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data" / "incoming"
OUTPUT_DIR = BASE_DIR / "artifacts" / "ingestion" / "identities" / "20261120"

# Paramètres modifiables selon les fichiers
ENCODING = "utf-8-sig"
SEPARATOR = None       # None = détection automatique ; sinon "," ";" ou "\t"
DATE_COLUMNS = []      # Exemple : ["timestamp", "date_mesure"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
# Détecter le séparateur et lire un CSV
def detect_separator(file_path: Path, encoding: str = ENCODING) -> str:
    """Détecte simplement le séparateur parmi , ; tabulation et |."""
    sample = file_path.read_text(
        encoding=encoding,
        errors="replace"
    )[:8192]

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
        return dialect.delimiter
    except csv.Error:
        return ","


def read_csv_file(file_path: Path) -> pd.DataFrame:
    """Lit un CSV avec les paramètres définis dans la configuration."""
    separator = SEPARATOR or detect_separator(file_path)

    df = pd.read_csv(
        file_path,
        sep=separator,
        encoding=ENCODING,
        low_memory=False
    )

    # Convertit uniquement les colonnes de dates indiquées dans DATE_COLUMNS
    for column in DATE_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")

    return df

In [12]:
# Fonctions de description et de statistiques
def get_column_type(series: pd.Series) -> str:
    """Retourne un type lisible pour la carte d'identité."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return "DATETIME"
    elif pd.api.types.is_numeric_dtype(series):
        return "NUMERIC"
    else:
        return "STRING"


def get_column_information(df: pd.DataFrame) -> list[dict]:
    """Construit la liste des colonnes et de leurs types."""
    return [
        {
            "Colname": column,
            "Inferred Type": get_column_type(df[column])
        }
        for column in df.columns
    ]


def get_column_statistics(df: pd.DataFrame) -> list[dict]:
    """Calcule les statistiques des colonnes numériques et dates."""
    statistics = []

    for column in df.columns:
        series = df[column].dropna()

        if series.empty:
            continue

        if pd.api.types.is_numeric_dtype(series):
            statistics.append({
                "Col name": column,
                "Min value": series.min(),
                "Max value": series.max(),
                "Mean value": round(series.mean(), 2),
                "Median value": round(series.median(), 2)
            })

        elif pd.api.types.is_datetime64_any_dtype(series):
            statistics.append({
                "Col name": column,
                "Min value": series.min(),
                "Max value": series.max(),
                "Mean value": "-",
                "Median value": "-"
            })

    return statistics

In [13]:
# Générer une carte d’identité
def create_identity(file_path: Path) -> Path:
    """Lit un CSV, génère sa carte Markdown et retourne le chemin créé."""
    process_begin = datetime.now()

    df = read_csv_file(file_path)

    file_size_kb = file_path.stat().st_size / 1024
    rows, cols = df.shape

    column_types = get_column_information(df)
    statistics = get_column_statistics(df)

    process_end = datetime.now()

    report = [
        f"# Carte d'identité : {file_path.name}",
        "",
        f"- Source : {file_path.resolve()}",
        f"- Filesize : {file_size_kb:.2f} Ko",
        f"- ProcessBeginAt : {process_begin:%H:%M:%S}",
        f"- ProcessEndAt : {process_end:%H:%M:%S}",
        f"- Charset : {ENCODING}",
        f"- Rows Number : {rows}",
        f"- Cols Number : {cols}",
        "",
        "## Column list",
        "",
        "| Colname | Inferred Type |",
        "|---|---|"
    ]

    for item in column_types:
        report.append(
            f"| {item['Colname']} | {item['Inferred Type']} |"
        )

    report.extend([
        "",
        "## Column statistics",
        "",
        "| Col name | Min value | Max value | Mean value | Median value |",
        "|---|---|---|---|---|"
    ])

    if statistics:
        for item in statistics:
            report.append(
                f"| {item['Col name']} "
                f"| {item['Min value']} "
                f"| {item['Max value']} "
                f"| {item['Mean value']} "
                f"| {item['Median value']} |"
            )
    else:
        report.append("| Aucune colonne numérique ou date détectée | - | - | - | - |")

    timestamp = process_end.strftime("%H%M%S")
    output_file = OUTPUT_DIR / f"{file_path.stem}_{timestamp}.md"

    output_file.write_text("\n".join(report), encoding="utf-8")

    return output_file

In [14]:
 #Traiter un fichier ou tous les CSV
csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    print("Aucun fichier CSV trouvé dans :", DATA_DIR)
else:
    for file_path in csv_files:
        identity_file = create_identity(file_path)
        print("Fiche générée :", identity_file)

Fiche générée : /home/iwanka/dev/formation_ia/sprint1/indusense/artifacts/ingestion/identities/20261120/releves_incidents_150334.md
Fiche générée : /home/iwanka/dev/formation_ia/sprint1/indusense/artifacts/ingestion/identities/20261120/telemetry_150335.md


In [15]:
from datetime import datetime
from pathlib import Path

import sqlglot
from sqlglot import exp

SQL_FILE = DATA_DIR / "machine.sql"

# Remplacez par le dialecte réel si vous le connaissez :
# exemples : "postgres", "mysql", "sqlite", "spark"
SQL_DIALECT = None

In [16]:
#Nouvelle cellule — Lire et parser le fichier SQL
process_begin = datetime.now()

sql_text = SQL_FILE.read_text(encoding="utf-8")

# Produit une liste d'instructions analysées.
# Si SQL_DIALECT vaut None, SQLGlot utilise son dialecte générique.
statements = sqlglot.parse(sql_text, read=SQL_DIALECT)

# Retire les éventuelles entrées vides
statements = [statement for statement in statements if statement is not None]

process_end = datetime.now()

print("Fichier :", SQL_FILE.resolve())
print("Instructions SQL détectées :", len(statements))

Fichier : /home/iwanka/dev/formation_ia/sprint1/indusense/datas/machine.sql
Instructions SQL détectées : 10


In [17]:
# Résumer les instructions et tables
statement_types = []
tables = set()

for statement in statements:
    statement_types.append(type(statement).__name__)

    for table in statement.find_all(exp.Table):
        tables.add(table.sql())

In [18]:
print("Types d'instructions :", statement_types)
print("Tables repérées :", sorted(tables))

Types d'instructions : ['Transaction', 'Create', 'Create', 'Create', 'Insert', 'Create', 'Create', 'Create', 'Insert', 'Commit']
Tables repérées : ['machine', 'maintenance']


In [19]:
#Générer la carte Markdown
file_size_kb = SQL_FILE.stat().st_size / 1024
line_count = len(sql_text.splitlines())

report = [
    f"# Carte d'identité : {SQL_FILE.name}",
    "",
    f"- Source : {SQL_FILE.resolve()}",
    f"- Filesize : {file_size_kb:.2f} Ko",
    f"- ProcessBeginAt : {process_begin:%H:%M:%S}",
    f"- ProcessEndAt : {process_end:%H:%M:%S}",
    "- Charset utilisé pour la lecture : UTF-8",
    f"- Rows Number : {line_count} lignes de texte",
    f"- SQL statements parsed : {len(statements)}",
    f"- SQL dialect : {SQL_DIALECT or 'générique'}",
    "",
    "## Types d'instructions détectés",
    "",
]

if statement_types:
    for statement_type in statement_types:
        report.append(f"- {statement_type}")
else:
    report.append("- Aucune instruction détectée")

report.extend([
    "",
    "## Tables repérées",
    ""
])

if tables:
    for table in sorted(tables):
        report.append(f"- `{table}`")
else:
    report.append("- Aucune table repérée")

report.extend([
    "",
    "> Le script a été parsé, mais il n'a pas été exécuté. "
    "Le parsing ne constitue pas une validation complète par un moteur SQL."
])

timestamp = process_end.strftime("%H%M%S")
output_file = OUTPUT_DIR / f"{SQL_FILE.stem}_{timestamp}.md"

output_file.write_text("\n".join(report), encoding="utf-8")

print("Fiche SQL générée :", output_file)

Fiche SQL générée : /home/iwanka/dev/formation_ia/sprint1/indusense/artifacts/ingestion/identities/20261120/machine_150335.md
